# BT4012 Kaggle 2026 — Illicit Bitcoin Transaction Detection

**Task:** predict the probability that each test transaction is illicit. **Metric:** ROC AUC.

**Decisions so far**
- **Validation (A1):** three expanding-window forward folds, because test time steps (36–49) all come after training (1–35).
- **Stage 1 model choice (B3):** Random Forest vs default LightGBM on the raw features; LightGBM won (public LB 0.94664).
- **Selection rule (C1):** after stage 1, fold F1 (validate on steps 21–27) matched the public LB (0.9451 vs 0.9466) while F2/F3 were over-optimistic (~0.996).
  Experiments are therefore ranked by **F1 AUC**, with mean fold AUC as tie-break.
- **Stage 2 (D1):** graph features from the transaction edge list.

*AI use: Claude (Claude Code) was used to write boilerplate code for this notebook; validation, feature and model choices were made by me.*

In [1]:
import time
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

SEED = 42
DATA = Path('.')                 # folder holding train.csv, test.csv, txs_edgelist.csv, sample_submission.csv
SUB_DIR = Path('submissions'); SUB_DIR.mkdir(exist_ok=True)
LOG = Path('experiment_log.csv')
np.random.seed(SEED)

## 1. Load data

- `label` is blank for unknown transactions. Unknown is **not** licit, so those rows are excluded from supervised training and evaluation.
- Features are read as float32 to halve memory.

In [2]:
feat_cols = [f'feat_{i}' for i in range(1, 166)]
dtypes = {c: 'float32' for c in feat_cols}

train = pd.read_csv(DATA / 'train.csv', dtype=dtypes)
test = pd.read_csv(DATA / 'test.csv', dtype=dtypes)
edges = pd.read_csv(DATA / 'txs_edgelist.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')

lab = train[train['label'].notna()].copy()
lab['label'] = lab['label'].astype(int)

print('train', train.shape, '| labelled', lab.shape, '| test', test.shape, '| edges', edges.shape)
print('illicit rate (labelled):', round(lab['label'].mean(), 4))
print('train steps', train.time_step.min(), '-', train.time_step.max(), '| test steps', test.time_step.min(), '-', test.time_step.max())
assert (test['index'].values == sample['index'].values).all(), 'test order differs from sample submission'

train (141772, 168) | labelled (31235, 168) | test (15329, 168) | edges (234355, 2)
illicit rate (labelled): 0.1167
train steps 1 - 35 | test steps 36 - 49


## 2. Validation design (A1): forward-in-time folds

Each fold trains only on earlier steps and validates on later ones, mimicking the train → test gap.
`time_step` is used only for splitting, never as a model feature (test values 36–49 never occur in training).

In [3]:
FOLDS = {
    'F1': (range(1, 21), range(21, 28)),
    'F2': (range(1, 28), range(28, 36)),
    'F3': (range(1, 25), range(25, 36)),   # long horizon, closest to the 14-step test window
}
for name, (tr_steps, va_steps) in FOLDS.items():
    va = lab[lab.time_step.isin(va_steps)]
    print(f'{name}: train {len(lab[lab.time_step.isin(tr_steps)]):>6} rows | valid {len(va):>5} rows, {va.label.sum()} illicit')

F1: train  18889 rows | valid  6034 rows, 686 illicit
F2: train  24923 rows | valid  6312 rows, 1187 illicit
F3: train  23606 rows | valid  7629 rows, 1425 illicit


## 3. Graph structure checks

An edge `txId1 → txId2` means an output of `txId1` is spent by `txId2`. Checks that shape the feature design:
- every edge stays within one time step, and there are no train ↔ test edges, so each step is its own graph;
- the graph is acyclic (funds only flow forward);
- **in the test period, unlabelled transactions appear in the edge list but have no feature rows**, whereas in training they do.
  Any feature built from *neighbours' features* must therefore only use neighbours that would also be visible in the test period (labelled train rows and test rows).

In [4]:
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))

step = pd.concat([train[['txId', 'time_step']], test[['txId', 'time_step']]]).set_index('txId')['time_step']
e_step = edges.assign(s1=edges.txId1.map(step), s2=edges.txId2.map(step)).dropna()
train_ids, test_ids = set(train.txId), set(test.txId)

print('directed acyclic graph:', nx.is_directed_acyclic_graph(G))
print('edges crossing time steps:', int((e_step.s1 != e_step.s2).sum()))
print('train <-> test edges:', int((edges.txId1.isin(train_ids) & edges.txId2.isin(test_ids)).sum()
                                   + (edges.txId1.isin(test_ids) & edges.txId2.isin(train_ids)).sum()))
print('edge-list nodes without a feature row:', G.number_of_nodes() - len(set(G.nodes) & (train_ids | test_ids)))

directed acyclic graph: True
edges crossing time steps: 0


train <-> test edges: 0
edge-list nodes without a feature row: 46668


## 4. Feature engineering (D1)

### 4a. Structural features (full edge list)
Computed from the full edge list, which covers every transaction in both periods, so they are built identically for train and test.

| Feature | Meaning |
| --- | --- |
| `in_deg`, `out_deg`, `deg_ratio` | observed funding transactions in / spending transactions out; fan-out vs fan-in |
| `is_source`, `is_sink` | no observed inputs / outputs within the graph |
| `in2_paths`, `out2_paths` | number of 2-hop paths into / out of the transaction |
| `depth_up`, `depth_down` | longest chain of transactions leading into / out of it |
| `n_anc`, `n_desc` | number of transactions upstream / downstream (all reachable) |
| `chain_run` | length of a run of 1-in, 1–2-out transactions ending here (peel-chain-like pattern) |

In [5]:
t0 = time.time()
gf = pd.DataFrame(index=pd.Index(step.index, name='txId'))
in_deg = edges.txId2.value_counts()
out_deg = edges.txId1.value_counts()
gf['in_deg'] = in_deg.reindex(gf.index).fillna(0)
gf['out_deg'] = out_deg.reindex(gf.index).fillna(0)
gf['deg_ratio'] = (gf.out_deg + 1) / (gf.in_deg + 1)
gf['is_source'] = (gf.in_deg == 0).astype(int)
gf['is_sink'] = (gf.out_deg == 0).astype(int)
gf['in2_paths'] = edges.assign(d=edges.txId1.map(in_deg).fillna(0)).groupby('txId2').d.sum().reindex(gf.index).fillna(0)
gf['out2_paths'] = edges.assign(d=edges.txId2.map(out_deg).fillna(0)).groupby('txId1').d.sum().reindex(gf.index).fillna(0)

# longest path in / out and peel-chain runs via one pass in topological order
order = list(nx.topological_sort(G))
depth_up, chain = {}, {}
for v in order:
    preds = list(G.predecessors(v))
    depth_up[v] = max((depth_up[u] + 1 for u in preds), default=0)
    chain_like = len(preds) == 1 and 1 <= G.out_degree(v) <= 2
    chain[v] = chain[preds[0]] + 1 if chain_like else 0
depth_down = {}
for v in reversed(order):
    depth_down[v] = max((depth_down[w] + 1 for w in G.successors(v)), default=0)

gf['depth_up'] = gf.index.map(depth_up).fillna(0)
gf['depth_down'] = gf.index.map(depth_down).fillna(0)
gf['chain_run'] = gf.index.map(chain).fillna(0)
gf['n_anc'] = [len(nx.ancestors(G, n)) if n in G else 0 for n in gf.index]
gf['n_desc'] = [len(nx.descendants(G, n)) if n in G else 0 for n in gf.index]

struct_cols = list(gf.columns)
print(f'{len(struct_cols)} structural features in {time.time() - t0:.1f}s')
gf.describe().T.round(2)

12 structural features in 27.1s


,count,mean,std,min,25%,50%,75%,max
in_deg,157101.0,1.24,4.34,0.0,0.0,1.0,1.0,284.0
out_deg,157101.0,1.16,1.94,0.0,1.0,1.0,1.0,472.0
deg_ratio,157101.0,1.35,1.34,0.0,1.0,1.0,2.0,236.5
is_source,157101.0,0.27,0.44,0.0,0.0,0.0,1.0,1.0
is_sink,157101.0,0.18,0.39,0.0,0.0,0.0,0.0,1.0
in2_paths,157101.0,1.30,6.06,0.0,0.0,1.0,1.0,326.0
out2_paths,157101.0,1.24,4.29,0.0,0.0,1.0,1.0,478.0
depth_up,157101.0,26.36,91.45,0.0,0.0,2.0,11.0,1248.0
depth_down,157101.0,25.00,89.07,0.0,1.0,3.0,10.0,1248.0
chain_run,157101.0,20.10,88.04,0.0,0.0,0.0,3.0,1247.0


### 4b. Neighbour feature aggregates (visibility-masked)

For each transaction: mean and max of selected features over its **funding** (in) neighbours and its **spending** (out) neighbours, plus how many such neighbours were visible.
Only labelled-train and test transactions count as neighbours, so training rows see their neighbourhood the same way test rows do (Section 3).
Neighbour **labels** are never used: test neighbours have no labels, so that would not transfer.

To keep the feature count manageable, aggregates are built for the 20 raw features most important to a LightGBM model trained on F1's training steps only (1–20), so the selection never sees any validation step.

In [6]:
tr1 = lab[lab.time_step.isin(FOLDS['F1'][0])]
imp_model = LGBMClassifier(random_state=SEED, verbose=-1, importance_type='gain').fit(tr1[feat_cols], tr1['label'])
top_feats = pd.Series(imp_model.feature_importances_, index=feat_cols).nlargest(20).index.tolist()
print('aggregated features:', top_feats)

visible = pd.concat([lab[['txId'] + top_feats], test[['txId'] + top_feats]]).set_index('txId')
ve = edges[edges.txId1.isin(visible.index) & edges.txId2.isin(visible.index)]

in_agg = ve.join(visible, on='txId1').groupby('txId2')[top_feats].agg(['mean', 'max'])
in_agg.columns = [f'in_{f}_{s}' for f, s in in_agg.columns]
out_agg = ve.join(visible, on='txId2').groupby('txId1')[top_feats].agg(['mean', 'max'])
out_agg.columns = [f'out_{f}_{s}' for f, s in out_agg.columns]

nf = pd.DataFrame(index=gf.index)
nf['n_vis_in'] = ve.txId2.value_counts().reindex(nf.index).fillna(0)
nf['n_vis_out'] = ve.txId1.value_counts().reindex(nf.index).fillna(0)
nf = nf.join(in_agg).join(out_agg)           # NaN when no visible neighbour; LightGBM handles NaN natively
nbr_cols = list(nf.columns)
print(len(nbr_cols), 'neighbour features')

aggregated features: ['feat_90', 'feat_53', 'feat_47', 'feat_2', 'feat_163', 'feat_41', 'feat_132', 'feat_100', 'feat_16', 'feat_5', 'feat_80', 'feat_76', 'feat_49', 'feat_59', 'feat_8', 'feat_58', 'feat_138', 'feat_142', 'feat_119', 'feat_103']


82 neighbour features


In [7]:
lab_x = lab.join(gf, on='txId').join(nf, on='txId')
test_x = test.join(gf, on='txId').join(nf, on='txId')

FEATURE_SETS = {
    'base': feat_cols,
    'graph': feat_cols + struct_cols,
    'graph_nbr': feat_cols + struct_cols + nbr_cols,
}
{k: len(v) for k, v in FEATURE_SETS.items()}

{'base': 165, 'graph': 177, 'graph_nbr': 259}

## 5. Models

In [8]:
def make_model(name):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=500, max_features='sqrt', n_jobs=-1, random_state=SEED)
    if name == 'lgbm':
        return LGBMClassifier(random_state=SEED, verbose=-1)
    raise ValueError(name)

## 6. Experiments

| id | stage | model | features |
| --- | --- | --- | --- |
| exp01b | B3 comparison | Random Forest | base |
| exp01 | B3 winner (submitted) | LightGBM defaults | base |
| exp02 | D1 | LightGBM defaults | base + structural |
| exp03 | D1 | LightGBM defaults | base + structural + neighbour aggregates |

All LightGBM runs keep default parameters so differences come from the features alone.

In [9]:
EXPERIMENTS = [
    ('exp01b', 'rf', 'base'),
    ('exp01', 'lgbm', 'base'),
    ('exp02', 'lgbm', 'graph'),
    ('exp03', 'lgbm', 'graph_nbr'),
]

def run_cv(model_name, cols):
    res = {}
    for fold, (tr_steps, va_steps) in FOLDS.items():
        tr = lab_x[lab_x.time_step.isin(tr_steps)]
        va = lab_x[lab_x.time_step.isin(va_steps)]
        model = make_model(model_name).fit(tr[cols], tr['label'])
        res[fold] = roc_auc_score(va['label'], model.predict_proba(va[cols])[:, 1])
    return res

rows = []
for exp_id, m, fs in EXPERIMENTS:
    t0 = time.time()
    res = run_cv(m, FEATURE_SETS[fs])
    rows.append({'exp_id': exp_id, 'model': m, 'features': fs, **res, 'sec': round(time.time() - t0, 1)})
    print(exp_id, m, fs, {k: round(v, 4) for k, v in res.items()})

summary = pd.DataFrame(rows).set_index('exp_id')
summary['mean'] = summary[list(FOLDS)].mean(axis=1)
summary['min'] = summary[list(FOLDS)].min(axis=1)
summary = summary.sort_values(['F1', 'mean'], ascending=False)    # selection rule C1
summary.round(4)

exp01b rf base {'F1': 0.8781, 'F2': 0.9953, 'F3': 0.9953}


exp01 lgbm base {'F1': 0.9451, 'F2': 0.9964, 'F3': 0.9965}


exp02 lgbm graph {'F1': 0.9434, 'F2': 0.9961, 'F3': 0.9963}


exp03 lgbm graph_nbr {'F1': 0.9599, 'F2': 0.9968, 'F3': 0.996}


,model,features,F1,F2,F3,sec,mean,min
exp_id,,,,,,,,
exp03,lgbm,graph_nbr,0.9599,0.9968,0.9960,14.1,0.9842,0.9599
exp01,lgbm,base,0.9451,0.9964,0.9965,11.9,0.9793,0.9451
exp02,lgbm,graph,0.9434,0.9961,0.9963,9.0,0.9786,0.9434
exp01b,rf,base,0.8781,0.9953,0.9953,44.7,0.9562,0.8781


## 7. Fit the selected experiment on all labelled steps (1–35) and write its submission

Selection rule C1: highest F1 AUC, mean fold AUC as tie-break. Writing a file does not mean it gets submitted; that is decided after reviewing the table above.

In [10]:
best = summary.index[0]
best_model, best_fs = summary.loc[best, 'model'], summary.loc[best, 'features']
cols = FEATURE_SETS[best_fs]
print('Selected:', best, best_model, best_fs)

final = make_model(best_model).fit(lab_x[cols], lab_x['label'])
sub = sample[['index']].copy()
sub['target'] = final.predict_proba(test_x[cols])[:, 1]

# schema checks against the template
assert sub.shape == sample.shape and list(sub.columns) == list(sample.columns)
assert (sub['index'].values == sample['index'].values).all()
assert np.isfinite(sub['target']).all() and sub['target'].between(0, 1).all()

sub_path = SUB_DIR / f'{best}_{best_model}_{best_fs}.csv'
sub.to_csv(sub_path, index=False)
print('saved', sub_path)

Selected: exp03 lgbm graph_nbr


saved submissions\exp03_lgbm_graph_nbr.csv


In [11]:
# which features drive the selected model (gain importance)
imp = pd.Series(final.booster_.feature_importance('gain'), index=cols) if best_model == 'lgbm' \
      else pd.Series(final.feature_importances_, index=cols)
imp = (imp / imp.sum()).sort_values(ascending=False)
print('share of importance from structural features:', round(imp.reindex(struct_cols).sum(), 3) if best_fs != 'base' else 0)
print('share of importance from neighbour features:', round(imp.reindex(nbr_cols).sum(), 3) if best_fs == 'graph_nbr' else 0)
imp.head(20).round(4)

share of importance from structural features: 0.051
share of importance from neighbour features: 0.088


feat_53            0.2645
feat_90            0.1140
feat_55            0.0952
feat_5             0.0785
feat_163           0.0472
chain_run          0.0461
feat_2             0.0310
in_feat_59_mean    0.0224
feat_132           0.0160
out_feat_59_max    0.0155
feat_59            0.0137
feat_46            0.0136
feat_76            0.0131
in_feat_53_mean    0.0120
feat_80            0.0120
feat_138           0.0115
feat_22            0.0112
feat_103           0.0088
feat_16            0.0068
feat_41            0.0062
dtype: float64

## 8. Experiment log

In [12]:
prev = pd.read_csv(LOG) if LOG.exists() else pd.DataFrame(columns=['exp_id', 'public_lb'])
log = pd.DataFrame([{
    'exp_id': e, 'model': r['model'], 'features': r['features'], 'seed': SEED,
    'params': str(make_model(r['model']).get_params()),
    **{f'auc_{f}': round(r[f], 4) for f in FOLDS},
    'cv_mean': round(r['mean'], 4), 'cv_min': round(r['min'], 4),
} for e, r in summary.iterrows()])
log = log.merge(prev[['exp_id', 'public_lb']], on='exp_id', how='left')   # keep recorded public scores
log['submission'] = log.exp_id.map({best: str(sub_path)})
log = pd.concat([prev[~prev.exp_id.isin(log.exp_id)], log])     # keep rows written by later sections
log = log.sort_values('exp_id')
log.to_csv(LOG, index=False)
log[['exp_id', 'model', 'features', 'auc_F1', 'auc_F2', 'auc_F3', 'cv_mean', 'public_lb', 'submission']]

,exp_id,model,features,auc_F1,auc_F2,auc_F3,cv_mean,public_lb,submission
1,exp01,lgbm,base,0.9451,0.9964,0.9965,0.9793,0.94664,NaN
3,exp01b,rf,base,0.8781,0.9953,0.9953,0.9562,NaN,NaN
2,exp02,lgbm,graph,0.9434,0.9961,0.9963,0.9786,NaN,NaN
0,exp03,lgbm,graph_nbr,0.9599,0.9968,0.9960,0.9842,0.95462,submissions\exp03_lgbm_graph_nbr.csv
4,exp04_w10,lgbm,graph_nbr,0.8801,0.9978,0.9974,0.9584,NaN,NaN
5,exp04_w14,lgbm,graph_nbr,0.8927,0.9978,0.9965,0.9623,NaN,NaN
6,exp04_w7,lgbm,graph_nbr,0.8980,0.9954,0.9972,0.9636,NaN,NaN
7,exp05_d02,lgbm,graph_nbr,0.9407,0.9972,0.9962,0.9781,NaN,NaN
8,exp05_d05,lgbm,graph_nbr,0.9386,0.9970,0.9964,0.9774,NaN,NaN
9,exp05_d10,lgbm,graph_nbr,0.9439,0.9962,0.9962,0.9788,NaN,NaN


## 9. Training window and time-decay weighting (experiment block 2)

**Hypothesis.** Illicit behaviour drifts over time (F1 vs F2/F3 gap, Section 6), so older steps may teach patterns that no longer hold.
Two ways to down-weight the past are tested on the exp03 feature set with default LightGBM, so only the training data changes:

- **Window:** train only on the last *k* steps before each fold's validation period (k = 7, 10, 14; "all" = exp03).
- **Time decay:** train on all steps with sample weight $w_i = e^{-\lambda (T - t_i)}$, where $T$ is the fold's last training step (λ = 0.02, 0.05, 0.1; half-lives ≈ 35, 14, 7 steps).

**Evaluation.** Same three forward folds; rule C1 (F1 first, mean as tie-break). Because F1 holds only 686 illicit transactions, each candidate's F1 gain over exp03 is checked with a paired bootstrap (resampling F1 validation rows, 500 draws).

**Acceptance.** Adopt a variant only if F1 improves by ≥ 0.003, the bootstrap shows the gain is positive in ≥ 90% of draws, and the fold mean does not fall.

**Cost.** 7 configurations × 3 folds of default LightGBM ≈ 1–2 minutes; no new features.

In [13]:
WIN_COLS = FEATURE_SETS['graph_nbr']
WINDOW_EXPS = {                       # exp_id: (window in steps or None = all, decay lambda or None)
    'exp03':     (None, None),        # reference: all steps, unweighted
    'exp04_w7':  (7, None),
    'exp04_w10': (10, None),
    'exp04_w14': (14, None),
    'exp05_d02': (None, 0.02),
    'exp05_d05': (None, 0.05),
    'exp05_d10': (None, 0.10),
}

def run_cv_window(window, decay):
    res, f1_pred = {}, None
    for fold, (tr_steps, va_steps) in FOLDS.items():
        last = max(tr_steps)
        tr = lab_x[lab_x.time_step.isin(tr_steps)]
        if window is not None:
            tr = tr[tr.time_step > last - window]
        weight = np.exp(-decay * (last - tr.time_step.values)) if decay is not None else None
        va = lab_x[lab_x.time_step.isin(va_steps)]
        model = make_model('lgbm').fit(tr[WIN_COLS], tr['label'], sample_weight=weight)
        pred = model.predict_proba(va[WIN_COLS])[:, 1]
        res[fold] = roc_auc_score(va['label'], pred)
        if fold == 'F1':
            f1_pred = pred
    return res, f1_pred

win_rows, f1_preds = [], {}
for exp_id, (window, decay) in WINDOW_EXPS.items():
    res, f1_preds[exp_id] = run_cv_window(window, decay)
    win_rows.append({'exp_id': exp_id, 'window': window or 'all', 'decay': decay or 0, **res})
    print(exp_id, {k: round(v, 4) for k, v in res.items()})

exp03 {'F1': 0.9599, 'F2': 0.9968, 'F3': 0.996}


exp04_w7 {'F1': 0.898, 'F2': 0.9954, 'F3': 0.9972}


exp04_w10 {'F1': 0.8801, 'F2': 0.9978, 'F3': 0.9974}


exp04_w14 {'F1': 0.8927, 'F2': 0.9978, 'F3': 0.9965}


exp05_d02 {'F1': 0.9407, 'F2': 0.9972, 'F3': 0.9962}


exp05_d05 {'F1': 0.9386, 'F2': 0.997, 'F3': 0.9964}


exp05_d10 {'F1': 0.9439, 'F2': 0.9962, 'F3': 0.9962}


In [14]:
# paired bootstrap of the F1 AUC gain over exp03
f1_y = lab_x[lab_x.time_step.isin(FOLDS['F1'][1])]['label'].values
rng = np.random.default_rng(SEED)
boots = [rng.integers(0, len(f1_y), len(f1_y)) for _ in range(500)]
ref = f1_preds['exp03']
for r in win_rows:
    p = f1_preds[r['exp_id']]
    diffs = np.array([roc_auc_score(f1_y[b], p[b]) - roc_auc_score(f1_y[b], ref[b]) for b in boots])
    r['F1_gain'] = r['F1'] - win_rows[0]['F1']
    r['P(gain>0)'] = (diffs > 0).mean()

win = pd.DataFrame(win_rows).set_index('exp_id')
win['mean'] = win[list(FOLDS)].mean(axis=1)
win['min'] = win[list(FOLDS)].min(axis=1)
base_mean = win.loc['exp03', 'mean']
win['accepted'] = (win.F1_gain >= 0.003) & (win['P(gain>0)'] >= 0.9) & (win['mean'] >= base_mean)
win.loc['exp03', 'accepted'] = False
win.sort_values(['F1', 'mean'], ascending=False).round(4)

,window,decay,F1,F2,F3,F1_gain,P(gain>0),mean,min,accepted
exp_id,,,,,,,,,,
exp03,all,0.00,0.9599,0.9968,0.9960,0.0000,0.0,0.9842,0.9599,False
exp05_d10,all,0.10,0.9439,0.9962,0.9962,-0.0160,0.0,0.9788,0.9439,False
exp05_d02,all,0.02,0.9407,0.9972,0.9962,-0.0192,0.0,0.9781,0.9407,False
exp05_d05,all,0.05,0.9386,0.9970,0.9964,-0.0213,0.0,0.9774,0.9386,False
exp04_w7,7,0.00,0.8980,0.9954,0.9972,-0.0619,0.0,0.9636,0.8980,False
exp04_w14,14,0.00,0.8927,0.9978,0.9965,-0.0672,0.0,0.9623,0.8927,False
exp04_w10,10,0.00,0.8801,0.9978,0.9974,-0.0798,0.0,0.9584,0.8801,False


In [15]:
# log the new configurations (replacing any earlier rows with the same ids)
new_log = pd.DataFrame([{
    'exp_id': e, 'model': 'lgbm', 'features': 'graph_nbr', 'seed': SEED,
    'params': f"defaults; window={r['window']}; decay={r['decay']}",
    **{f'auc_{f}': round(r[f], 4) for f in FOLDS},
    'cv_mean': round(r['mean'], 4), 'cv_min': round(r['min'], 4),
} for e, r in win.drop(index='exp03').iterrows()])
log = pd.read_csv(LOG)
new_log = new_log.merge(log[['exp_id', 'public_lb']], on='exp_id', how='left')   # keep recorded public scores
log = pd.concat([log[~log.exp_id.isin(new_log.exp_id)], new_log]).sort_values('exp_id')
log.to_csv(LOG, index=False)
log[['exp_id', 'features', 'params', 'auc_F1', 'auc_F2', 'auc_F3', 'cv_mean', 'public_lb']]

,exp_id,features,params,auc_F1,auc_F2,auc_F3,cv_mean,public_lb
0,exp01,base,"{'boosting_type': 'gbdt', 'class_weight': None...",0.9451,0.9964,0.9965,0.9793,0.94664
1,exp01b,base,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.8781,0.9953,0.9953,0.9562,NaN
2,exp02,graph,"{'boosting_type': 'gbdt', 'class_weight': None...",0.9434,0.9961,0.9963,0.9786,NaN
3,exp03,graph_nbr,"{'boosting_type': 'gbdt', 'class_weight': None...",0.9599,0.9968,0.9960,0.9842,0.95462
1,exp04_w10,graph_nbr,defaults; window=10; decay=0.0,0.8801,0.9978,0.9974,0.9584,NaN
2,exp04_w14,graph_nbr,defaults; window=14; decay=0.0,0.8927,0.9978,0.9965,0.9623,NaN
0,exp04_w7,graph_nbr,defaults; window=7; decay=0.0,0.8980,0.9954,0.9972,0.9636,NaN
3,exp05_d02,graph_nbr,defaults; window=all; decay=0.02,0.9407,0.9972,0.9962,0.9781,NaN
4,exp05_d05,graph_nbr,defaults; window=all; decay=0.05,0.9386,0.9970,0.9964,0.9774,NaN
5,exp05_d10,graph_nbr,defaults; window=all; decay=0.1,0.9439,0.9962,0.9962,0.9788,NaN


## 10. Two-stage stacking: neighbour risk scores (experiment block 3)

**Hypothesis.** A neighbour's *predicted* illicit risk summarises all of its features in one number, so aggregating it may carry more signal than the 20-feature neighbour means/maxes of Section 4b.

**Method.**
1. **Model A** (default LightGBM) scores every visible transaction.
   Training rows get **out-of-fold** scores from 5 folds grouped by time step. Every edge stays within one step and each step forms essentially one connected component, so a transaction and all its neighbours are always scored by a model that saw none of their labels.
   Validation/test rows get scores from model A fitted on all training rows, which is exactly what happens at prediction time.
2. For each transaction, mean and max of model A scores over its funding (in), spending (out) and all neighbours. As in Section 4b, only labelled-train/test rows count as neighbours.
3. **Model B** (default LightGBM) = exp03 features + these 6 scores.

The whole procedure is rerun inside each validation fold, so validation labels never reach model A or the score features.
Two versions of model A: **exp06a** on raw features only; **exp06b** on the exp03 feature set.

**Acceptance.** Same as block 2: F1 gain ≥ 0.003 over exp03, positive in ≥ 90% of paired bootstrap draws, fold mean not lower.

**Cost.** Per fold: 6 model-A fits + 1 model-B fit ≈ 7× a single fit; ≈ 2–4 minutes for both versions.

In [16]:
from sklearn.model_selection import GroupKFold

SCORE_COLS = ['nb_in_mean', 'nb_in_max', 'nb_out_mean', 'nb_out_max', 'nb_all_mean', 'nb_all_max']
STACK_COLS = FEATURE_SETS['graph_nbr'] + SCORE_COLS

def stage_a_scores(fit_rows, pred_rows, a_cols, n_splits=5):
    # out-of-fold scores for fit_rows (folds grouped by time step) + full-fit scores for pred_rows
    oof = np.zeros(len(fit_rows))
    for tr_i, va_i in GroupKFold(n_splits=n_splits).split(fit_rows, groups=fit_rows.time_step):
        m = make_model('lgbm').fit(fit_rows.iloc[tr_i][a_cols], fit_rows.iloc[tr_i]['label'])
        oof[va_i] = m.predict_proba(fit_rows.iloc[va_i][a_cols])[:, 1]
    m = make_model('lgbm').fit(fit_rows[a_cols], fit_rows['label'])
    full = m.predict_proba(pred_rows[a_cols])[:, 1]
    return pd.concat([pd.Series(oof, index=fit_rows.txId.values), pd.Series(full, index=pred_rows.txId.values)])

def neighbour_scores(score):
    # mean / max of model-A scores over in-, out- and all neighbours that have a score
    e = edges[edges.txId1.isin(score.index) & edges.txId2.isin(score.index)]
    into = pd.DataFrame({'node': e.txId2.values, 's': e.txId1.map(score).values})   # funders of node
    outof = pd.DataFrame({'node': e.txId1.values, 's': e.txId2.map(score).values})  # spenders of node
    out = pd.DataFrame(index=pd.Index(score.index, name='txId'))
    for name, frame in [('in', into), ('out', outof), ('all', pd.concat([into, outof]))]:
        agg = frame.groupby('node').s.agg(['mean', 'max']).reindex(out.index)
        out[f'nb_{name}_mean'], out[f'nb_{name}_max'] = agg['mean'].values, agg['max'].values
    return out                                                                       # NaN = no scored neighbour

STACK_EXPS = {'exp06a': feat_cols, 'exp06b': FEATURE_SETS['graph_nbr']}           # model A feature sets

def run_cv_stack(a_cols):
    res, f1_pred = {}, None
    for fold, (tr_steps, va_steps) in FOLDS.items():
        tr = lab_x[lab_x.time_step.isin(tr_steps)]
        va = lab_x[lab_x.time_step.isin(va_steps)]
        nbs = neighbour_scores(stage_a_scores(tr, va, a_cols))
        tr_b, va_b = tr.join(nbs, on='txId'), va.join(nbs, on='txId')
        model_b = make_model('lgbm').fit(tr_b[STACK_COLS], tr_b['label'])
        pred = model_b.predict_proba(va_b[STACK_COLS])[:, 1]
        res[fold] = roc_auc_score(va_b['label'], pred)
        if fold == 'F1':
            f1_pred = pred
    return res, f1_pred

stack_rows = [{'exp_id': 'exp03', 'model_A': '-', **win.loc['exp03', list(FOLDS)].to_dict()}]
for exp_id, a_cols in STACK_EXPS.items():
    t0 = time.time()
    res, f1_preds[exp_id] = run_cv_stack(a_cols)
    stack_rows.append({'exp_id': exp_id, 'model_A': 'raw' if a_cols is feat_cols else 'graph_nbr', **res})
    print(exp_id, {k: round(v, 4) for k, v in res.items()}, f'{time.time() - t0:.0f}s')

exp06a {'F1': 0.9427, 'F2': 0.9974, 'F3': 0.9974} 77s


exp06b {'F1': 0.9512, 'F2': 0.9975, 'F3': 0.9977} 103s


In [17]:
# same paired bootstrap and acceptance rule as block 2
for r in stack_rows:
    p = f1_preds[r['exp_id']]
    diffs = np.array([roc_auc_score(f1_y[b], p[b]) - roc_auc_score(f1_y[b], ref[b]) for b in boots])
    r['F1_gain'] = r['F1'] - stack_rows[0]['F1']
    r['P(gain>0)'] = (diffs > 0).mean()

stack = pd.DataFrame(stack_rows).set_index('exp_id')
stack['mean'] = stack[list(FOLDS)].mean(axis=1)
stack['min'] = stack[list(FOLDS)].min(axis=1)
stack['accepted'] = (stack.F1_gain >= 0.003) & (stack['P(gain>0)'] >= 0.9) & (stack['mean'] >= stack.loc['exp03', 'mean'])
stack.loc['exp03', 'accepted'] = False
stack.sort_values(['F1', 'mean'], ascending=False).round(4)

,model_A,F1,F2,F3,F1_gain,P(gain>0),mean,min,accepted
exp_id,,,,,,,,,
exp03,-,0.9599,0.9968,0.9960,0.0000,0.0,0.9842,0.9599,False
exp06b,graph_nbr,0.9512,0.9975,0.9977,-0.0087,0.0,0.9821,0.9512,False
exp06a,raw,0.9427,0.9974,0.9974,-0.0172,0.0,0.9792,0.9427,False


### 10b. Final fit of the best accepted stacking variant (only if one is accepted)

Model A out-of-fold scores over all labelled steps 1–35 (grouped by step), model A on all steps scores the test set, then model B on all labelled steps.

In [18]:
accepted = stack[stack.accepted].sort_values(['F1', 'mean'], ascending=False)
stack_sub_path = None
if len(accepted):
    best_stack = accepted.index[0]
    a_cols = STACK_EXPS[best_stack]
    nbs = neighbour_scores(stage_a_scores(lab_x, test_x, a_cols))
    lab_b, test_b = lab_x.join(nbs, on='txId'), test_x.join(nbs, on='txId')
    model_b = make_model('lgbm').fit(lab_b[STACK_COLS], lab_b['label'])

    sub = sample[['index']].copy()
    sub['target'] = model_b.predict_proba(test_b[STACK_COLS])[:, 1]
    assert sub.shape == sample.shape and list(sub.columns) == list(sample.columns)
    assert (sub['index'].values == sample['index'].values).all()
    assert np.isfinite(sub['target']).all() and sub['target'].between(0, 1).all()
    stack_sub_path = SUB_DIR / f'{best_stack}_lgbm_stack.csv'
    sub.to_csv(stack_sub_path, index=False)
    print('selected', best_stack, '-> saved', stack_sub_path)

    imp = pd.Series(model_b.booster_.feature_importance('gain'), index=STACK_COLS)
    print('share of importance from neighbour-score features:', round(imp[SCORE_COLS].sum() / imp.sum(), 3))
else:
    print('no stacking variant met the acceptance rule; exp03 remains the best model')

no stacking variant met the acceptance rule; exp03 remains the best model


In [19]:
new_log = pd.DataFrame([{
    'exp_id': e, 'model': 'lgbm 2-stage', 'features': 'graph_nbr + nb_scores', 'seed': SEED,
    'params': f"defaults; model A on {r['model_A']}; OOF grouped by time step (5 folds)",
    **{f'auc_{f}': round(r[f], 4) for f in FOLDS},
    'cv_mean': round(r['mean'], 4), 'cv_min': round(r['min'], 4),
    'submission': str(stack_sub_path) if stack_sub_path and e == stack_sub_path.stem.split('_')[0] else '',
} for e, r in stack.drop(index='exp03').iterrows()])
log = pd.read_csv(LOG)
new_log = new_log.merge(log[['exp_id', 'public_lb']], on='exp_id', how='left')   # keep recorded public scores
log = pd.concat([log[~log.exp_id.isin(new_log.exp_id)], new_log]).sort_values('exp_id')
log.to_csv(LOG, index=False)
log[['exp_id', 'features', 'auc_F1', 'auc_F2', 'auc_F3', 'cv_mean', 'public_lb', 'submission']]

,exp_id,features,auc_F1,auc_F2,auc_F3,cv_mean,public_lb,submission
0,exp01,base,0.9451,0.9964,0.9965,0.9793,0.94664,NaN
1,exp01b,base,0.8781,0.9953,0.9953,0.9562,NaN,NaN
2,exp02,graph,0.9434,0.9961,0.9963,0.9786,NaN,NaN
3,exp03,graph_nbr,0.9599,0.9968,0.9960,0.9842,0.95462,submissions\exp03_lgbm_graph_nbr.csv
4,exp04_w10,graph_nbr,0.8801,0.9978,0.9974,0.9584,NaN,NaN
5,exp04_w14,graph_nbr,0.8927,0.9978,0.9965,0.9623,NaN,NaN
6,exp04_w7,graph_nbr,0.8980,0.9954,0.9972,0.9636,NaN,NaN
7,exp05_d02,graph_nbr,0.9407,0.9972,0.9962,0.9781,NaN,NaN
8,exp05_d05,graph_nbr,0.9386,0.9970,0.9964,0.9774,NaN,NaN
9,exp05_d10,graph_nbr,0.9439,0.9962,0.9962,0.9788,NaN,NaN


### 10c. Diagnostic submission: exp06b (not a selected model)

exp06b is the first candidate on which the folds disagree: F1 is lower than exp03 (−0.009) but F2/F3 are higher (+0.001/+0.002).
Its public score tests **which fold predicts the leaderboard**, which decides how all remaining experiments are judged:
- public clearly **below** exp03 (0.95462) → F1 is the right yardstick; keep rule C1;
- public clearly **above** exp03 → F1 alone is too pessimistic; switch to a rule that weights F2/F3.

Expected: about 0.948 if F1 tracks the leaderboard, about 0.956+ if F2/F3 do. Differences under about 0.003 count as inconclusive (public LB noise).

In [20]:
diag_id = 'exp06b'
nbs = neighbour_scores(stage_a_scores(lab_x, test_x, STACK_EXPS[diag_id]))
lab_b, test_b = lab_x.join(nbs, on='txId'), test_x.join(nbs, on='txId')
model_b = make_model('lgbm').fit(lab_b[STACK_COLS], lab_b['label'])

sub = sample[['index']].copy()
sub['target'] = model_b.predict_proba(test_b[STACK_COLS])[:, 1]
assert sub.shape == sample.shape and list(sub.columns) == list(sample.columns)
assert (sub['index'].values == sample['index'].values).all()
assert np.isfinite(sub['target']).all() and sub['target'].between(0, 1).all()
diag_path = SUB_DIR / f'{diag_id}_lgbm_stack_DIAGNOSTIC.csv'
sub.to_csv(diag_path, index=False)

ref_sub = pd.read_csv(SUB_DIR / 'exp03_lgbm_graph_nbr.csv')
print('saved', diag_path)
print('rank correlation with exp03 submission:', round(sub.target.corr(ref_sub.target, method='spearman'), 4))

log = pd.read_csv(LOG)
log.loc[log.exp_id == diag_id, 'submission'] = str(diag_path)
log.to_csv(LOG, index=False)

saved submissions\exp06b_lgbm_stack_DIAGNOSTIC.csv
rank correlation with exp03 submission: 0.9418


## 11. Four-fold validation, selection rule R2 and the exp03 + exp06b ensemble

**What the diagnostic showed.** Public scores: exp01 0.94664 → exp03 0.95462 → exp06b 0.95873.
The exp01 → exp03 gain showed up on F1; the exp03 → exp06b gain showed up on F2/F3 while F1 fell. No single fold (nor the 3-fold mean) ranks all three correctly, so rule C1 (F1 first) is retired.
The public difference between exp03 and exp06b (+0.004) is within public-leaderboard noise (30% of test rows), so both are treated as strong candidates with different strengths.

**Changes.**
1. **Fourth fold F4:** train 1–30, validate 31–35, the most recent observable period, adding a second "near-future" check alongside F2.
2. **Rule R2:** accept a candidate over a reference only if its **mean AUC over F1–F4 is higher** and it **loses no more than 0.005 on any single fold**. It rejects large single-period failures (training windows lost 0.06–0.08 on F1). Note it also rejects exp06b against exp03 (F1 −0.009), even though exp06b scored higher on the public leaderboard; see the results below.
3. **Ensemble exp07:** equal-weight average of the normalised ranks of exp03 and exp06b. ROC AUC depends only on ranking, so averaging ranks combines the two without needing calibrated probabilities.

**Cost.** exp03 and exp06b re-scored on 4 folds (≈ 3 minutes); the ensemble itself is free.

In [21]:
from scipy.stats import rankdata

ALL_FOLDS = {**FOLDS, 'F4': (range(1, 31), range(31, 36))}
va4 = {f: lab_x[lab_x.time_step.isin(v)] for f, (_, v) in ALL_FOLDS.items()}
y4 = {f: va4[f]['label'].values for f in ALL_FOLDS}
print({f: f'{len(y)} rows, {y.sum()} illicit' for f, y in y4.items()})

def fold_preds_single(cols):
    # exp03-style model: one LightGBM on the given columns, per fold
    out = {}
    for fold, (tr_steps, _) in ALL_FOLDS.items():
        tr = lab_x[lab_x.time_step.isin(tr_steps)]
        out[fold] = make_model('lgbm').fit(tr[cols], tr['label']).predict_proba(va4[fold][cols])[:, 1]
    return out

def fold_preds_stack(a_cols):
    # exp06-style two-stage model, rebuilt inside each fold (Section 10)
    out = {}
    for fold, (tr_steps, _) in ALL_FOLDS.items():
        tr, va = lab_x[lab_x.time_step.isin(tr_steps)], va4[fold]
        nbs = neighbour_scores(stage_a_scores(tr, va, a_cols))
        tr_b, va_b = tr.join(nbs, on='txId'), va.join(nbs, on='txId')
        out[fold] = make_model('lgbm').fit(tr_b[STACK_COLS], tr_b['label']).predict_proba(va_b[STACK_COLS])[:, 1]
    return out

def rank_average(*preds):
    # equal-weight mean of normalised ranks (0-1]
    return np.mean([rankdata(p) / len(p) for p in preds], axis=0)

t0 = time.time()
preds4 = {'exp03': fold_preds_single(FEATURE_SETS['graph_nbr']), 'exp06b': fold_preds_stack(STACK_EXPS['exp06b'])}
preds4['exp07'] = {f: rank_average(preds4['exp03'][f], preds4['exp06b'][f]) for f in ALL_FOLDS}
print(f'scored in {time.time() - t0:.0f}s')

{'F1': '6034 rows, 686 illicit', 'F2': '6312 rows, 1187 illicit', 'F3': '7629 rows, 1425 illicit', 'F4': '4330 rows, 690 illicit'}


scored in 124s


In [22]:
auc4 = pd.DataFrame({e: {f: roc_auc_score(y4[f], p[f]) for f in ALL_FOLDS} for e, p in preds4.items()}).T
auc4['mean'] = auc4[list(ALL_FOLDS)].mean(axis=1)
auc4['min'] = auc4[list(ALL_FOLDS)].min(axis=1)
auc4['public_lb'] = auc4.index.map({'exp03': 0.95462, 'exp06b': 0.95873})

def rule_r2(cand, ref, max_drop=0.005):
    # True if cand beats ref on 4-fold mean and loses <= max_drop on every fold
    diff = auc4.loc[cand, list(ALL_FOLDS)] - auc4.loc[ref, list(ALL_FOLDS)]
    return bool(auc4.loc[cand, 'mean'] > auc4.loc[ref, 'mean'] and diff.min() >= -max_drop), round(diff.min(), 4)

checks = pd.DataFrame([{'candidate': c, 'reference': r, 'accepted (R2)': rule_r2(c, r)[0], 'worst fold change': rule_r2(c, r)[1]}
                       for c, r in [('exp06b', 'exp03'), ('exp07', 'exp03'), ('exp07', 'exp06b')]])
display(auc4.round(4))
checks

,F1,F2,F3,F4,mean,min,public_lb
exp03,0.9599,0.9968,0.9960,0.9985,0.9878,0.9599,0.9546
exp06b,0.9512,0.9975,0.9977,0.9988,0.9863,0.9512,0.9587
exp07,0.9560,0.9975,0.9973,0.9989,0.9874,0.9560,NaN


,candidate,reference,accepted (R2),worst fold change
0,exp06b,exp03,False,-0.0087
1,exp07,exp03,False,-0.0039
2,exp07,exp06b,True,-0.0004


### 11b. Ensemble submission file

The ensemble of the two full-data models is the rank average of their submission files (both regenerated earlier in this notebook: exp03 in Section 7, exp06b in Section 10c).

In [23]:
s03 = pd.read_csv(SUB_DIR / 'exp03_lgbm_graph_nbr.csv')
s06 = pd.read_csv(SUB_DIR / 'exp06b_lgbm_stack_DIAGNOSTIC.csv')
assert (s03['index'].values == sample['index'].values).all() and (s06['index'].values == sample['index'].values).all()

sub = sample[['index']].copy()
sub['target'] = rank_average(s03.target.values, s06.target.values)
assert np.isfinite(sub['target']).all() and sub['target'].between(0, 1).all()
ens_path = SUB_DIR / 'exp07_rankavg_exp03_exp06b.csv'
sub.to_csv(ens_path, index=False)
print('saved', ens_path)

# log: add F4 / 4-fold mean for the re-scored experiments and a row for the ensemble
log = pd.read_csv(LOG)
for e in ['exp03', 'exp06b']:
    log.loc[log.exp_id == e, 'auc_F4'] = round(auc4.loc[e, 'F4'], 4)
    log.loc[log.exp_id == e, 'cv_mean4'] = round(auc4.loc[e, 'mean'], 4)
log = log[log.exp_id != 'exp07']
log = pd.concat([log, pd.DataFrame([{
    'exp_id': 'exp07', 'model': 'rank average', 'features': 'exp03 + exp06b', 'seed': SEED,
    'params': 'equal-weight mean of normalised ranks',
    **{f'auc_{f}': round(auc4.loc['exp07', f], 4) for f in ALL_FOLDS},
    'cv_mean4': round(auc4.loc['exp07', 'mean'], 4), 'cv_min': round(auc4.loc['exp07', 'min'], 4),
    'submission': str(ens_path),
}])]).sort_values('exp_id')
log.to_csv(LOG, index=False)
log.loc[log.exp_id.isin(['exp03', 'exp06b', 'exp07']), ['exp_id', 'auc_F1', 'auc_F2', 'auc_F3', 'auc_F4', 'cv_mean4', 'public_lb']]

saved submissions\exp07_rankavg_exp03_exp06b.csv


,exp_id,auc_F1,auc_F2,auc_F3,auc_F4,cv_mean4,public_lb
3,exp03,0.9599,0.9968,0.9960,0.9985,0.9878,0.95462
11,exp06b,0.9512,0.9975,0.9977,0.9988,0.9863,0.95873
0,exp07,0.9560,0.9975,0.9973,0.9989,0.9874,NaN
